In [ ]:
'''
extracting all light curves and saving them in csv files
per image

'''

In [22]:
import numpy as np
import pandas as pd
from astropy.io import fits
import os
import review_findings_functions as rff
import glob

In [ ]:
#let's figure out where the x,y positions of the stars are

In [15]:
xy = fits.open('/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020148032_000686_zg_c09_x_1.fits')

In [12]:
xy[1].data

FITS_rec([(2413., 2555.), (2333.,  127.), (2546., 1432.), ( 341.,  885.),
          (1199., 1460.), (1544., 2883.), ( 151., 2675.), (  18., 1036.),
          (1454., 2806.), ( 756.,  193.), (1011., 1973.), ( 389.,  841.),
          ( 977., 2113.), ( 135., 2902.), ( 265., 2865.), (1217., 1554.),
          (1590., 2095.), (1687., 1925.), (1099., 1546.), (1387., 2794.),
          (2973., 1839.), (2825., 2620.), (2223., 1749.), (3044., 2367.),
          (1109., 1470.), (1737., 2504.), (1765., 1288.), (2855., 2693.),
          (2654., 1993.), (1752., 2539.), (1118.,  535.), (2588., 2139.),
          (2092., 2379.), (1140.,  736.), ( 216.,  666.), (2077., 2268.),
          (  87., 1372.), (1659., 1540.), (2433., 1855.), (1955., 2303.),
          ( 459., 1540.)],
         dtype=(numpy.record, [('X', '>f8'), ('Y', '>f8')]))

In [16]:
xy_positions = load_xy_positions_from_fits('/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020148032_000686_zg_c09_x_1.fits')

Found binary table in extension 1
Loaded 41 positions from /lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020148032_000686_zg_c09_x_1.fits
X range: 18.0 - 3044.0
Y range: 127.0 - 2902.0


In [19]:
xy_positions = []

In [20]:
np.shape(xy_positions)[0]

0

In [110]:
xy_file = '/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020148032_000686_zg_c09_x_1.fits'

In [114]:
image_path[-47:-13]

'ztf_20221020148032_000686_zg_c09_x'

In [8]:
image_path = '/lustre/lrspec/ZTF_data/science_extractions/to_extract/1020_minus_extras/ztf_20221020148032_000686_zg_c09_x_0001.fits.fz'

light curves already exist - be aware this does not check for completion
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020163819_000640_zg_c14_x_0020
light curves already exist - be aware this does not check for completion
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020138241_000686_zi_c13_x_0000
Found binary table in extension 1
Loaded 61 positions from /lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020141806_000686_zi_c10_x_1.fits
X range: 6.0 - 3046.0
Y range: 153.0 - 2890.0
Loading FITS file: /lustre/lrspec/ZTF_data/science_extractions/to_extract/1020_minus_extras/ztf_20221020141806_000686_zi_c10_x_0001.fits.fz
Processing 61 positions...
Calculating background...
Image shape: (3080, 3072)
Background value: 171.25
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020141806_000686_zi_c10_x_0001_1_1826_2485.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020141806_000686_zi_c10_x_0001_1_2252_18

KeyboardInterrupt: 

In [11]:
def process_xy_positions_efficient_test(xy_positions, image_path, quad):
    """
    Efficiently process multiple x,y positions by loading the image once
    
    Parameters:
    -----------
    xy_positions : array-like
        Array of (x, y) coordinates, shape (N, 2)
    image_path : str
        Path to FITS file
    quad : int
        Quadrant number
    tracking : bool
        Whether to track streak position
    output_csv : str
        Output CSV filename
        
    Returns:
    --------
    nothing, saves light curves
    """
    xy_positions = np.array(xy_positions)
    
    filename = image_path[-47:-8]
    
    print(f"Loading FITS file: {image_path}")
    print(f"Processing {len(xy_positions)} positions...")
    
    # Load image once
    try:
        hdul = fits.open(image_path)
        file1_array = hdul[quad].data
        hdul.close()
        
        # Calculate background once
        print("Calculating background...")
        background = rff.get_image_background(file1_array)
        imagenobackground = file1_array - background
        
        print(f"Image shape: {imagenobackground.shape}")
        print(f"Background value: {background:.2f}")
        
    except Exception as e:
        print(f"Error loading FITS file: {e}")
        return None
    
    # Process all positions
    
    for i, (x_pos, y_pos) in enumerate(xy_positions):
        try:
            tot_2,tot_3,tot_4,tot_5, maxcols = get_tot_from_xy_efficient(
                imagenobackground, int(x_pos), int(y_pos), tracking=True)
            
            # Create DataFrame for this position
            position_df = pd.DataFrame({
                'position_id': i,
                'x_input': x_pos,
                'y_input': y_pos,
                'row_index': range(len(tot_2)),
                'flux_hap_2': tot_2,
                'flux_hap_3': tot_3,
                'flux_hap_4': tot_4,
                'flux_hap_5': tot_5,
                'max_col': maxcols
            })
            
            

            individual_filename = f"/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_{filename}_{quad}_{int(x_pos):04}_{int(y_pos):04}.csv"
            print("saving")
            print(individual_filename)
            position_df.to_csv(individual_filename, index=False)

                
        except Exception as e:
            print(f"Error processing position {i} ({x_pos}, {y_pos}): {e}")
            continue
    
    return position_df

In [4]:
def get_tot_from_xy_efficient(imagenobackground, x_pos, y_pos, tracking=True):
    """
    Extract flux from a single x,y position using pre-loaded image data
    
    Parameters:
    -----------
    imagenobackground : array
        Pre-loaded and background-subtracted image data
    x_pos : int
        X coordinate (column)
    y_pos : int
        Y coordinate (row)
    half_aperture : int
        Half-width of aperture
    tracking : bool
        Whether to track streak position
        
    Returns:
    --------
    tot : array
        Flux values for each row
    maxcols : array
        Column positions used for each row
    """
    if tracking == False:
        # Fixed position approach
        zeropoint = x_pos - 7
        search_start = max(0, zeropoint)
        search_end = min(imagenobackground.shape[1], zeropoint + 14)
        
        if search_end > search_start and y_pos - 10 >= 0 and y_pos + 10 < imagenobackground.shape[0]:
            averages = np.average(
                imagenobackground[y_pos - 10 : y_pos + 10, search_start : search_end], axis=0
            )
            maxind = np.argmax(averages)
            maxcol = search_start + maxind
        else:
            maxcol = x_pos
            
        tot = np.zeros(imagenobackground.shape[0])
        
        for r in range(imagenobackground.shape[0]):
            combined = imagenobackground[r, maxcol] if maxcol < imagenobackground.shape[1] else 0
            for i in range(1, half_aperture + 1):
                if maxcol - i >= 0:
                    combined += imagenobackground[r, maxcol - i]
                if maxcol + i < imagenobackground.shape[1]:
                    combined += imagenobackground[r, maxcol + i]
            tot[r] = combined
        
        return tot, np.full(len(tot), maxcol)
    
    else:
        # Tracking approach
        tot_2 = np.zeros(imagenobackground.shape[0])
        tot_3 = np.zeros(imagenobackground.shape[0])
        tot_4 = np.zeros(imagenobackground.shape[0])
        tot_5 = np.zeros(imagenobackground.shape[0])
        maxcols = np.zeros(imagenobackground.shape[0])
        zeropoint = x_pos - 10
        
        for r in range(imagenobackground.shape[0]):
            search_start = max(0, zeropoint)
            search_end = min(imagenobackground.shape[1], zeropoint + 20)
            
            if search_end > search_start:
                maxcol = np.argmax(imagenobackground[r, search_start:search_end]) + search_start
            else:
                maxcol = x_pos
                
            maxcols[r] = maxcol
            combined = imagenobackground[r, maxcol] if maxcol < imagenobackground.shape[1] else 0
            
            half_aperture = 2
            for i in range(1, half_aperture + 1):
                if maxcol - i >= 0:
                    combined += imagenobackground[r, maxcol - i]
                if maxcol + i < imagenobackground.shape[1]:
                    combined += imagenobackground[r, maxcol + i]
            
            tot_2[r] = combined
            
            half_aperture = 3
            for i in range(1, half_aperture + 1):
                if maxcol - i >= 0:
                    combined += imagenobackground[r, maxcol - i]
                if maxcol + i < imagenobackground.shape[1]:
                    combined += imagenobackground[r, maxcol + i]
            
            tot_3[r] = combined
            
            half_aperture = 4
            for i in range(1, half_aperture + 1):
                if maxcol - i >= 0:
                    combined += imagenobackground[r, maxcol - i]
                if maxcol + i < imagenobackground.shape[1]:
                    combined += imagenobackground[r, maxcol + i]
            
            tot_4[r] = combined
            
            half_aperture = 5
            for i in range(1, half_aperture + 1):
                if maxcol - i >= 0:
                    combined += imagenobackground[r, maxcol - i]
                if maxcol + i < imagenobackground.shape[1]:
                    combined += imagenobackground[r, maxcol + i]
            
            tot_5[r] = combined
            
            
        return tot_2,tot_3,tot_4,tot_5, maxcols
    


In [5]:
def load_xy_positions_from_fits(bintable_path):
    """
    Load X,Y positions from FITS binary table
    
    Parameters:
    -----------
    bintable_path : str
        Path to FITS file containing binary table with X,Y coordinates
        
    Returns:
    --------
    xy_positions : array
        Array of (x, y) coordinates
    """
    try:
        with fits.open(bintable_path) as hdul:
            # Try to find the binary table (usually extension 1, but could vary)
            table_data = None
            for i, hdu in enumerate(hdul):
                if hasattr(hdu, 'data') and hdu.data is not None:
                    if hasattr(hdu.data, 'dtype') and len(hdu.data.dtype.names or []) > 0:
                        table_data = hdu.data
                        print(f"Found binary table in extension {i}")
                        break
            
            if table_data is None:
                print(f"No binary table found in {bintable_path}")
                return None
            
            # Extract X,Y coordinates
            # Handle different possible column names
            x_col = None
            y_col = None
            
            for col_name in table_data.dtype.names:
                if col_name.upper() in ['X', 'X_POS', 'XPOS', 'COL']:
                    x_col = col_name
                elif col_name.upper() in ['Y', 'Y_POS', 'YPOS', 'ROW']:
                    y_col = col_name
            
            if x_col is None or y_col is None:
                print(f"Could not find X,Y columns in {bintable_path}")
                print(f"Available columns: {table_data.dtype.names}")
                return None
            
            # Extract positions
            x_vals = table_data[x_col]
            y_vals = table_data[y_col]
            xy_positions = np.column_stack([x_vals, y_vals])
            
            print(f"Loaded {len(xy_positions)} positions from {bintable_path}")
            print(f"X range: {x_vals.min():.1f} - {x_vals.max():.1f}")
            print(f"Y range: {y_vals.min():.1f} - {y_vals.max():.1f}")
            
            return xy_positions
            
    except Exception as e:
        print(f"Error loading positions from {bintable_path}: {e}")
        return None

In [127]:


image_directory = '/lustre/lrspec/ZTF_data/science_extractions/to_extract/1020_minus_extras/'
xy_directory = '/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/'
lightcurve_path = '/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_'

image_files = os.listdir(image_directory)

for image_filename in image_files:
    # Create full path to image file
    image_path = os.path.join(image_directory, image_filename)
    
    # Extract filename parts for bintable
    bintable_filename = image_filename[-47:-13]  
    
    filename = image_filename[-47:-8]  
    
    # Check if lightcurve files already exist
    if glob.glob(lightcurve_path + filename + '*'):  # Fixed: use glob instead of is_file
        print('light curves already exist - be aware this does not check for completion')
        print(lightcurve_path + filename)
        continue
    
    for quad in range(1, 5):
        bintable_path = xy_directory + bintable_filename + '_' + str(quad) + '.fits'  # Fixed: removed extra str() and parenthesis
        xy_positions = load_xy_positions_from_fits(bintable_path)
        process_xy_positions_efficient(xy_positions, image_path, quad)

light curves already exist - be aware this does not check for completion
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020163819_000640_zg_c14_x_0020
light curves already exist - be aware this does not check for completion
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020138241_000686_zi_c13_x_0000
light curves already exist - be aware this does not check for completion
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020141806_000686_zi_c10_x_0001
light curves already exist - be aware this does not check for completion
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020141806_000686_zi_c14_x_0025
Found binary table in extension 1
Loaded 28 positions from /lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020163819_000640_zg_c10_x_1.fits
X range: 55.0 - 3064.0
Y range: 215.0 - 2959.0
Loading FITS file: /lustre/lrspec/ZTF_data/science_extractions/to_extract/1020_minus_extras/ztf_20221020163819_000640_zg_c10_x_0004.fits

Traceback (most recent call last):
  File "/lustre/lrspec/users/3155/anaconda3/envs/trans/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3505, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_37459/1723266020.py", line 25, in <module>
    process_xy_positions_efficient(xy_positions, image_path, quad)
  File "/tmp/ipykernel_37459/3466799171.py", line 51, in process_xy_positions_efficient
    tot_2,tot_3,tot_4,tot_5, maxcols = get_tot_from_xy_efficient(
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_37459/3439727074.py", line -1, in get_tot_from_xy_efficient
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/lustre/lrspec/users/3155/anaconda3/envs/trans/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 2102, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^

In [2]:
csv = pd.read_csv('/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020163819_000640_zg_c10_x_0004_2_1547_2057.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020163819_000640_zg_c10_x_0004_2_1547_2057.csv'

In [12]:
df = process_xy_positions_efficient_test(xy_positions, image_path, quad=1)

Loading FITS file: /lustre/lrspec/ZTF_data/science_extractions/to_extract/1020_minus_extras/ztf_20221020148032_000686_zg_c09_x_0001.fits.fz
Processing 41 positions...
Calculating background...
Image shape: (3080, 3072)
Background value: 153.68
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020148032_000686_zg_c09_x_0001_1_2413_2555.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020148032_000686_zg_c09_x_0001_1_2333_0127.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020148032_000686_zg_c09_x_0001_1_2546_1432.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020148032_000686_zg_c09_x_0001_1_0341_0885.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020148032_000686_zg_c09_x_0001_1_1199_1460.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_20221020148032_000686_zg_c09_x_0001_1_1544_2883.csv
saving
/lustre/lrspec/ZTF_data/lightcurves/1020/lightcurve_ztf_202

In [13]:
df

,position_id,x_input,y_input,row_index,flux_hap_2,flux_hap_3,flux_hap_4,flux_hap_5,max_col
0,40,459.0,1540.0,0,44.260895,45.332550,37.406433,23.320221,459.0
1,40,459.0,1540.0,1,55.304260,82.199387,93.198624,98.283875,459.0
2,40,459.0,1540.0,2,47.141525,62.420593,62.399658,51.121368,459.0
3,40,459.0,1540.0,3,43.625977,44.127960,33.297882,15.909164,459.0
4,40,459.0,1540.0,4,62.890625,88.957550,111.031189,133.701523,460.0
...,...,...,...,...,...,...,...,...,...
3075,40,459.0,1540.0,3075,47.572784,64.789322,70.822281,67.774918,459.0
3076,40,459.0,1540.0,3076,45.108459,50.808121,44.697876,34.464447,459.0
3077,40,459.0,1540.0,3077,52.696045,67.681168,64.484421,58.479279,459.0
3078,40,459.0,1540.0,3078,62.870392,95.903351,123.825089,146.734573,459.0


In [30]:
image_filename[0:3]

'ztf'

In [26]:
type(xy_positions)

list